# 02 – Data Quality & Cleaning

**Input:** data/raw/meals_raw.csv (23,663 meals from fetch_mensa.py)

**Goal:** check data quality, decide which canteens to keep, and clean the data.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/meals_raw.csv")
df.shape

(23663, 12)

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23663 entries, 0 to 23662
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                23663 non-null  int64  
 1   name              23663 non-null  str    
 2   category          23663 non-null  str    
 3   notes             23663 non-null  str    
 4   canteen_id        23663 non-null  int64  
 5   canteen_name      23663 non-null  str    
 6   city              23663 non-null  str    
 7   date              23663 non-null  str    
 8   prices.students   19768 non-null  float64
 9   prices.employees  19768 non-null  float64
 10  prices.pupils     1331 non-null   float64
 11  prices.others     17996 non-null  float64
dtypes: float64(4), int64(2), str(6)
memory usage: 2.2 MB


In [3]:
df.head()

,id,name,category,notes,canteen_id,canteen_name,city,date,prices.students,prices.employees,prices.pupils,prices.others
0,20807890,Paniertes Seelachsfilet,Fischgericht,"['Dillsoße', 'Fingermöhren', 'Brokkoli', 'Kurk...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,4.10,6.15,NaN,8.15
1,20807891,Frikadelle mit Käse überbacken,Fleischgericht,"['Gebratenes Putensteak', 'Letscho ungarische ...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,3.70,5.95,NaN,7.85
2,20559265,Letscho ungarische Art,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75
3,20559266,Fingermöhren,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75
4,20559267,Karotten-Kürbis-Gemüse,Gemüsebeilage,"['Antioxidationsmittel', 'Süßungsmitteln', 'Se...",63,"Leipzig, Mensa am Park",Leipzig,2024-10-02,0.75,1.35,NaN,1.75


In [4]:
df.isna().sum()

id                      0
name                    0
category                0
notes                   0
canteen_id              0
canteen_name            0
city                    0
date                    0
prices.students      3895
prices.employees     3895
prices.pupils       22332
prices.others        5667
dtype: int64

## Coverage: does every canteen have enough data?
I requested one Wednesday per week from 2024-10-01 to 2026-10-01.

In [5]:
all_dates = pd.date_range("2024-10-01", "2026-10-01", freq="W-WED")
total_dates = len(all_dates)
print("Dates requested per canteen:", total_dates)

Dates requested per canteen: 105


In [6]:
coverage = (
    df.groupby(["city", "canteen_name"])["date"]
      .nunique()
      .reset_index(name="dates_with_meals")
)

coverage["coverage_pct"] = (coverage["dates_with_meals"] / total_dates * 100).round(1)
coverage.sort_values("coverage_pct")

,city,canteen_name,dates_with_meals,coverage_pct
31,Würzburg,"Würzburg, Mensa am Studentenhaus",3,2.9
12,Jena,Mensa Ernst-Abbe-Platz,20,19.0
13,Jena,Mensa Philosophenweg,20,19.0
26,Saarbrücken / Saarland,Cafeteria Hochschule für Musik Saar,37,35.2
6,Düsseldorf,Mensa Kamp-Lintfort,46,43.8
7,Düsseldorf,"Mensa Universitätsstraße, Düsseldorf",46,43.8
1,Berlin,Mensa TU Hardenbergstraße,47,44.8
0,Berlin,"Berlin, Wilhelmstadtschulen",64,61.0
11,Heidelberg,"Heidelberg, Triplex-Mensa am Uniplatz",65,61.9
22,Passau,Cafeteria Nikolakloster Passau,67,63.8


In [7]:
CITIES = ["Heidelberg", "Berlin", "Hamburg", "München", "Leipzig",
          "Würzburg", "Nürnberg", "Dresden", "Köln",
          "Mannheim", "Chemnitz", "Straubing", "Passau", "Regensburg",
          "Düsseldorf", "Stuttgart", "Saarbrücken / Saarland",
          "Dortmund / Nordrhein-Westfalen", "Jena"]

missing_cities = set(CITIES) - set(df["city"].unique())
missing_cities

{'Dortmund / Nordrhein-Westfalen', 'Hamburg', 'München'}

## Decision: which canteens do I keep?
Looking at the coverage table, there is a clear gap: 7 canteens are below 45%, and all others are above 60%.
Jena and Düsseldorf have identical counts for both canteens, which suggests their data stopped at some point.
Hamburg, München and Dortmund returned no data at all.
Two canteens are schools, not universities (Berlin Wilhelmstadtschulen, Straubing ABG Gymnasium).

My rule:
1. Keep canteens with at least 50% coverage, because it falls in the natural gap and removes canteens whose data stopped.
2. Remove school canteens, because this project is about university students.

In [8]:
MIN_COVERAGE = 50
SCHOOL_CANTEENS = ["Berlin, Wilhelmstadtschulen",
                   "Straubing, ABG Anton-Bruckner-Gymnasium"]

good_canteens = coverage[coverage["coverage_pct"] >= MIN_COVERAGE]["canteen_name"]
df_clean = df[df["canteen_name"].isin(good_canteens)]
df_clean = df_clean[~df_clean["canteen_name"].isin(SCHOOL_CANTEENS)]

print("Canteens kept:", df_clean["canteen_name"].nunique())
print("Cities kept:", df_clean["city"].nunique())
print("Meals kept:", len(df_clean))

Canteens kept: 23
Cities kept: 13
Meals kept: 17918


## Part B – Cleaning
Problems noted in notebook 01:
1. Text has stray line breaks (\n)
2. Notes are stored as text that looks like a list
3. prices.pupils is almost always empty
4. Some meals are priced per 100g, not per portion
5. Some meals have no student price

In [9]:
df_clean = df_clean.copy()

In [10]:
df_clean["name"] = df_clean["name"].str.replace("\n", " ").str.strip()
df_clean["category"] = df_clean["category"].str.strip()
df_clean["name"].head()

0           Paniertes Seelachsfilet
1    Frikadelle mit Käse überbacken
2            Letscho ungarische Art
3                      Fingermöhren
4            Karotten-Kürbis-Gemüse
Name: name, dtype: str

In [11]:
print(type(df_clean["notes"].iloc[0]))
print(df_clean["notes"].iloc[0])

<class 'str'>
['Dillsoße', 'Fingermöhren', 'Brokkoli', 'Kurkumareis', 'glutenhaltiges Getreide', 'Fisch', 'Weizen', 'Eier', 'Milch/ Milchzucker', 'Sulfit/ Schwefeldioxid', 'Fisch', 'Antioxidationsmittel']


In [12]:
import ast

df_clean["notes"] = df_clean["notes"].apply(ast.literal_eval)                        # text → real list
df_clean["notes"] = df_clean["notes"].apply(lambda lst: "; ".join(n.strip() for n in lst))  # list → clean text
df_clean["notes"].head()

0    Dillsoße; Fingermöhren; Brokkoli; Kurkumareis;...
1    Gebratenes Putensteak; Letscho ungarische Art;...
2    Antioxidationsmittel; Süßungsmitteln; Senf; Vegan
3    Antioxidationsmittel; Süßungsmitteln; Senf; Vegan
4    Antioxidationsmittel; Süßungsmitteln; Senf; Vegan
Name: notes, dtype: str

In [13]:
df_clean = df_clean.drop(columns=["prices.pupils"])
df_clean.columns

Index(['id', 'name', 'category', 'notes', 'canteen_id', 'canteen_name', 'city',
       'date', 'prices.students', 'prices.employees', 'prices.others'],
      dtype='str')

In [14]:
per_100g = df_clean["name"].str.contains(r"100\s?g", case=False, regex=True)

print("Meals priced per 100g:", per_100g.sum())
df_clean[per_100g][["canteen_name", "name", "prices.students"]].head(10)

Meals priced per 100g: 417


,canteen_name,name,prices.students
3003,"Dresden, Alte Mensa",100g gebrannte Mandeln,3.10
9475,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9480,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9486,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9492,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9498,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9504,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9510,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9516,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92
9522,"Heidelberg, Mensa Im Neuenheimer Feld 304",Schlemmerbuffet (je 100g) Reichhaltige Auswahl...,0.92


In [15]:
df[df.index.isin(per_100g[per_100g].index)]["canteen_name"].value_counts()

canteen_name
Köln, Mensa Zülpicher Straße                 162
Heidelberg, Mensa Im Neuenheimer Feld 304     95
Köln, Mensa Südstadt                          94
Heidelberg, Triplex-Mensa am Uniplatz         65
Dresden, Alte Mensa                            1
Name: count, dtype: int64

In [16]:
removed = df.loc[per_100g[per_100g].index]

removed.groupby("canteen_name")["prices.students"].median().sort_values()

canteen_name
Köln, Mensa Südstadt                         0.75
Köln, Mensa Zülpicher Straße                 0.75
Heidelberg, Mensa Im Neuenheimer Feld 304    0.92
Heidelberg, Triplex-Mensa am Uniplatz        0.99
Dresden, Alte Mensa                          3.10
Name: prices.students, dtype: float64

In [17]:
removed = removed.copy()
removed["in_name"] = removed["name"].str.contains(r"100\s?g", case=False, regex=True)
removed.groupby("canteen_name")["in_name"].mean().round(2)

canteen_name
Dresden, Alte Mensa                          1.0
Heidelberg, Mensa Im Neuenheimer Feld 304    1.0
Heidelberg, Triplex-Mensa am Uniplatz        1.0
Köln, Mensa Südstadt                         1.0
Köln, Mensa Zülpicher Straße                 1.0
Name: in_name, dtype: float64

In [18]:
df_clean = df_clean[~per_100g]

In [19]:
no_price = df_clean[df_clean["prices.students"].isna()]
print("Meals without student price:", len(no_price))
no_price["canteen_name"].value_counts()

Meals without student price: 258


canteen_name
Mensa / Mensacafe Saarbrücken                110
Chemnitz, Reichenhainer Straße, Mensa         70
Heidelberg, Triplex-Mensa am Uniplatz         29
Chemnitz, Straße der Nationen, Mensa          19
Heidelberg, Mensa Im Neuenheimer Feld 304     13
Köln, Mensa Zülpicher Straße                   5
Mensa Insel Schütt                             4
Mensa Regensburger Straße                      4
Dresden, Alte Mensa                            3
Köln, Mensa Südstadt                           1
Name: count, dtype: int64

In [20]:
no_price[["canteen_name", "name", "category"]].sample(10, random_state=1)

,canteen_name,name,category
12362,"Chemnitz, Reichenhainer Straße, Mensa","Neues Jahr, neues Glück“ - ""Schlag das StuWe!”...",zZz Hinweis zZz
8154,Mensa / Mensacafe Saarbrücken,Sommerhilfe,Information
12368,"Chemnitz, Reichenhainer Straße, Mensa","""Save the Date"" Nach dem Quiz ist vor dem Quiz...",zZz Hinweis zZz
12917,"Chemnitz, Straße der Nationen, Mensa",Die Mensa und Cafeteria Straße der Nationen bl...,Achtung
12669,"Chemnitz, Reichenhainer Straße, Mensa",Wir wünschen allen frohe Weihnachten und einen...,Frohe Festtage
12354,"Chemnitz, Reichenhainer Straße, Mensa",Das Team der mensa55 wünscht euch eine tolle W...,Hinweis
10398,"Heidelberg, Triplex-Mensa am Uniplatz",Eis,A
8097,Mensa / Mensacafe Saarbrücken,Sommerhilfe,Information
10330,"Heidelberg, Triplex-Mensa am Uniplatz",Pasta + More,Pasta Container
12383,"Chemnitz, Reichenhainer Straße, Mensa","Neues Jahr, neues Glück“ - ""Schlag das StuWe!”...",zZz Hinweis zZz


In [21]:
df_clean = df_clean[df_clean["prices.students"].notna()]

In [22]:
df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean.dtypes

id                           int64
name                           str
category                       str
notes                          str
canteen_id                   int64
canteen_name                   str
city                           str
date                datetime64[us]
prices.students            float64
prices.employees           float64
prices.others              float64
dtype: object

In [23]:
print("Final rows:", len(df_clean))
print("Canteens:", df_clean["canteen_name"].nunique())
df_clean.isna().sum()

Final rows: 17243
Canteens: 23


id                     0
name                   0
category               0
notes                  0
canteen_id             0
canteen_name           0
city                   0
date                   0
prices.students        0
prices.employees       0
prices.others       1771
dtype: int64

In [24]:
df_clean.to_csv("../data/processed/meals_clean.csv", index=False)

## Cleaning decisions
- **Per-100g meals:** My first rule searched the name and notes for "100g" and found 985 meals.
  Checking showed that in Stuttgart, Mannheim and Freiburg "100g" only appeared in nutrition notes
  ("Nährwerte pro 100 g"), not in the price. So I changed the rule to search the name only.
  Result: removed 417 meals priced per 100g (Heidelberg and Köln buffets, snacks in Dresden),
  because they can't be compared with per-portion prices.
- **No student price:** Removed 258 rows. Most were not meals but announcements posted as menu
  items (categories like "Information", "zZz Hinweis zZz", "Frohe Festtage"). A few real items were lost too, which I accept.
- **Text:** Removed line breaks (\n) from names and categories, and turned notes into readable text.
- **Columns:** Dropped

## Part C1 – Meal type classification
Goal: label every meal as vegan / vegetarian / meat / fish / unknown,
using keywords. The meal name is trusted first, the notes second.

In [25]:
df_clean["category"].value_counts().head(40)

category
Beilagen                   1646
Hauptgerichte              1454
Nachspeisen                1052
Pastateller                 777
Smoothie                    681
Schneller Teller            485
Suppe                       462
Dessert                     415
Wahlessen                   389
QUERBEET                    389
A+B                         378
Sättigungsbeilage           364
Menü 1                      303
Buffet                      280
Gemüsebeilage               261
Wok                         247
Pasta                       246
Vegan                       209
xX Cafeteria - Pizza Xx     205
Menü vegan                  202
Essen 1                     196
Essen 2                     195
Vegetarisch                 192
Wochenangebot               191
A                           178
xXx Cafeteria StrNa xXx     173
Aktion                      170
MEISTERWERK                 170
Veganes Gericht             155
Mensacafé                   147
Essen 3                     145

In [26]:
notes_words = (
    df_clean["notes"].str.split("; ").explode().str.strip().str.lower()
)
notes_words.value_counts().head(50)

notes
weizen                               5187
vegan                                4339
sellerie                             4061
vegetarisch                          4048
mit antioxidationsmittel             3101
senf                                 2814
milch und milchprodukte              2354
glutenhaltiges getreide              2352
eier                                 2162
soja                                 2112
antioxidationsmittel                 1993
= vegan                              1718
mit farbstoff                        1698
milch/ milchzucker                   1661
farbstoff                            1350
portion preis                        1282
konservierungsstoff                  1222
hafer                                1105
milch                                 994
glutenhaltiges getreide (a)           872
schwein                               863
weizen (a1)                           861
mit weizen                            795
milch/laktose               

In [27]:
VEGAN  = ["vegan"]
VEGGIE = ["vegetarisch", "vegetarian", "fleischlos", "veggie"]
MEAT   = ["fleisch", "schwein", r"\brind", "kalb", "geflügel", "hähnchen", r"\bhuhn",
          r"\bpute", r"\blamm", "speck", "schinken", "wurst", "würstchen", r"\bhack",
          "salami", "gyros", "döner", "chicken", "beef", "frikadelle", "gulasch",
          "jägerschnitzel"]
FISH   = ["fisch", "lachs", "thunfisch", "forelle", "hering", "garnele",
          "scampi", "kabeljau", "zander", "dorsch", "calamari", "tintenfisch"]

In [28]:
# Name + category: describes the dish itself → most reliable
name_text = (df_clean["category"] + " " + df_clean["name"]).str.lower()

# Notes: noisy (sides, allergens). Remove Mannheim's "= Vegan" legend.
notes_text = (
    df_clean["notes"]
    .str.lower()
    .str.replace(r"=\s*(vegan|vegetarisch)", "", regex=True)
)

In [29]:
def has(text, words):
    return text.str.contains("|".join(words), regex=True)

In [30]:
import numpy as np

conditions = [
    has(name_text, VEGAN),
    has(name_text, VEGGIE),
    has(name_text, MEAT),
    has(name_text, FISH),
    has(notes_text, MEAT),
    has(notes_text, FISH),
    has(notes_text, VEGAN),
    has(notes_text, VEGGIE),
]
choices = ["vegan", "vegetarian", "meat", "fish",
           "meat", "fish", "vegan", "vegetarian"]

df_clean["meal_type"] = np.select(conditions, choices, default="unknown")
df_clean["meal_type"].value_counts()

meal_type
vegan         5811
vegetarian    3998
meat          3898
unknown       3194
fish           342
Name: count, dtype: int64

### Checks
Keyword rules can make mistakes, so I checked the result in several ways:
random examples per meal type, the share of each type per canteen,
and a closer look at canteens with surprising results (Mannheim, Chemnitz, Stuttgart).

In [31]:
for t in ["vegan", "vegetarian", "meat", "fish", "unknown"]:
    print("=====", t)
    print(df_clean[df_clean["meal_type"] == t]["name"].sample(8, random_state=1).to_string())

===== vegan
16290    MG Süd RestaurantVEGAN: Green Oat Burger Pomme...
21298    Süßkartoffelpfanne mit Datteln, Tomaten-Gurken...
2389                                          Nach Angebot
15524      VEGAN: Kartoffel-Lauchcremesuppe Kaiserbrötchen
1125                                Mango Erdbeer Smoothie
6384                                         Pommes frites
5463                                 Petersilienkartoffeln
11471    Sellerieschnitzel , Kräutersoße , Zucchini, Pi...
===== vegetarian
2135                                Heidelbeer Buttermilch
12933    Hirtenkäse-Kichererbsen-Pfanne mit Tomaten, Bo...
4029     Rührei (G) mit Rahmspinat (G) und Petersilienk...
7290                            Schoko Mousse schwarz-weiß
22219    Kartoffelpuffer, Erbsen-Guacamole, Kräuter-Jog...
8965                        Spinatlasagne  mit Kräutersoße
4865                                      Gemischter Salat
7714                                        Amaretto Creme
===== meat
13269    Marienf

In [32]:
pd.crosstab(df_clean["canteen_name"], df_clean["meal_type"], normalize="index").round(2)

meal_type,fish,meat,unknown,vegan,vegetarian
canteen_name,,,,,
Cafeteria Nikolakloster Passau,0.04,0.12,0.00,0.45,0.39
"Chemnitz, Reichenhainer Straße, Mensa",0.03,0.32,0.35,0.02,0.28
"Chemnitz, Straße der Nationen, Mensa",0.04,0.25,0.45,0.00,0.26
"Dresden, Alte Mensa",0.05,0.37,0.03,0.33,0.23
"Dresden, Mensa Matrix",0.01,0.45,0.01,0.44,0.10
"Heidelberg, Mensa Im Neuenheimer Feld 304",0.00,0.03,0.85,0.13,0.00
"Heidelberg, Triplex-Mensa am Uniplatz",0.00,0.00,0.83,0.16,0.01
"Köln, Mensa Südstadt",0.00,0.23,0.18,0.53,0.06
"Köln, Mensa Zülpicher Straße",0.00,0.24,0.09,0.49,0.19


In [33]:
def look(canteen, mtype, n=10):
    rows = df_clean[(df_clean["canteen_name"] == canteen) & (df_clean["meal_type"] == mtype)]
    return rows[["name", "notes"]].sample(min(n, len(rows)), random_state=1)

def top_notes(canteen, n=25):
    notes = df_clean[df_clean["canteen_name"] == canteen]["notes"]
    return notes.str.split("; ").explode().str.strip().str.lower().value_counts().head(n)

In [34]:
notes_l = df_clean["notes"].str.lower()
has_eq_vegan  = notes_l.str.contains("= vegan")
has_eq_veggie = notes_l.str.contains("= vegetarisch")

check = pd.DataFrame({
    "canteen": df_clean["canteen_name"],
    "eq_vegan": has_eq_vegan,
    "both": has_eq_vegan & has_eq_veggie,
})
check.groupby("canteen")[["eq_vegan", "both"]].mean().round(2).sort_values("eq_vegan", ascending=False).head(5)

,eq_vegan,both
canteen,,
"Mannheim, Mensa am Schloss",0.97,0.62
"Mannheim, Mensaria Metropol",0.90,0.22
Cafeteria Nikolakloster Passau,0.00,0.00
"Dresden, Alte Mensa",0.00,0.00
"Dresden, Mensa Matrix",0.00,0.00


In [35]:
mannheim  = df_clean["canteen_name"].str.startswith("Mannheim")
meat_name = has(name_text, MEAT)

print("Mannheim meat dishes:", (mannheim & meat_name).sum())
print("...of those with '= Vegan':", has_eq_vegan[mannheim & meat_name].mean().round(2))

Mannheim meat dishes: 339
...of those with '= Vegan': 0.99


In [36]:
top_notes("Chemnitz, Reichenhainer Straße, Mensa")

notes
milch                       264
vegetarisch                 179
eier                        173
mit antioxidationsmittel    158
mit konservierungsstoff     153
mit knoblauch               148
soja                        138
mit farbstoff               108
senf                        100
sellerie                     69
mit süßungsmittel            56
                             49
sulfit/schwefel              45
schwein                      45
erdnüsse                     31
geflügel                     29
sesam                        28
mit phosphat                 27
rind                         22
mit alkohol                  19
krebstiere                   18
mit geschmacksverstärker     17
fisch                        16
mit karmin e120              15
mit honig                    14
Name: count, dtype: int64

In [37]:
top_notes("Stuttgart Nord, Mensa Kunstakademie")

notes
vegan                                                                                                                                                                                                          321
weizen                                                                                                                                                                                                         299
sellerie                                                                                                                                                                                                       241
milch und laktose                                                                                                                                                                                              218
schwefeldioxid so2 und sulfite                                                                                                                        

In [38]:
look("Stuttgart Nord, Mensa Kunstakademie", "unknown")

,name,notes
16812,Tomatensuppe,Sellerie
17066,Penne mit Pesto a la Genovese,Ei; Mandeln; Milch und Laktose; Weizen
17007,Beilagenbuffet,
16817,Sahneschokopudding,Milch und Laktose
17183,Beilagensalat,
16933,Panna Cotta mit Erdbeermus,Milch und Laktose
17305,Gnocchi in Spinatsauce mit gerösteten Sonnenbl...,Milch und Laktose; Sellerie; Weizen; mit Antio...
16903,Exotischer Obstsalat,Schwefeldioxid SO2 und Sulfite; mit Antioxidat...
16858,Broccolisuppe,Milch und Laktose; Sellerie
17138,Beilagensalat,


In [39]:
df_clean[name_text.str.contains("schnitzel")].groupby("meal_type")["name"].count()

meal_type
meat          268
unknown        22
vegan         100
vegetarian     48
Name: name, dtype: int64

In [40]:
df_clean[name_text.str.contains("schnitzel") & (df_clean["meal_type"] == "unknown")]["name"].head(15)

12277    Hausgemachtes Kohlrabischnitzel in Pankopanade...
12476    Hausgemachtes Kohlrabischnitzel an Bratenjus d...
12790    Hausgemachtes Kohlrabischnitzel an Dillsoße da...
12829    Gemüseschnitzel dazu mediterranes Ratatouilleg...
12833    Sächsisches Schnitzel in würziger Senfpanade a...
12838    Hausgemachtes Zucchinischnitzel an Schnittlauc...
12910    Hausgemachtes Zucchinischnitzel mit Schnittlau...
12935    Hausgemachtes Kohlrabischnitzel in Pankopanade...
13169    Hausgemachtes Kohlrabischnitzel an Bratenjus d...
13356    Schnitzel-Burger mit würzigem Bergkäse, Eisber...
13450    Schnitzel-Burger mit würzigem Bergkäse, Eisber...
13535    Schnitzel-Burger mit würzigem Bergkäse, Eisber...
13644    Hausgemachtes Kohlrabischnitzel an Dillsoße da...
13719    Gemüseschnitzel dazu mediterranes Ratatouilleg...
13731    Sächsisches Schnitzel in würziger Senfpanade a...
Name: name, dtype: str

### Classification decisions
- **Name first, notes second:** notes describe all components of a plate (e.g. vegan fries next to ham),
  so a "vegan" tag in notes made "Prager Schinken" vegan. Now the name is checked first.
- **Rule order matters:** "vegan" before meat words, so "veganer Hackbraten" stays vegan.
  "fleischlos" is in the vegetarian list, checked before meat.
- **Word boundaries (\b):** short words like "lamm" matched inside "Flammkuchen". Using \blamm fixed it.
- **Mannheim "= Vegan" legend:** 90–97% of Mannheim items had "= Vegan" in the notes. Test: 99% of dishes
  with meat in the name (e.g. "Cordon bleu vom Schwein") also had it. So it's an icon legend, not a label.
  Removed "= vegan" / "= vegetarisch" before classifying. No other canteen uses this format.
- **"schnitzel" not added as a meat word:** half of the untagged schnitzels are vegetable schnitzels
  (Kohlrabi, Zucchini). Only "jägerschnitzel" was added, which is always meat.

### Limitations
- Labels depend on how each canteen tags its food. I only use the canteens' own labels and don't guess.
- **Chemnitz** tags vegetarian but never vegan, so untagged dishes like tofu stir-fries stay "unknown".
- **Mannheim** has no real vegetarian label, so vegetarian dishes there are "unknown".
- **Heidelberg** barely tags anything, so most items are "unknown".
- **Stuttgart Kunstakademie** has 0% meat. Checked: it tags vegan/vegetarian, has no meat words in its notes,
  and its unknown items are soups, sides and desserts. So it's a real result (a meat-free canteen).
- Because labelling differs between canteens, comparing vegan shares between cities partly compares
  labelling habits. Price comparisons between meal types are best done **within the same canteen**.